# GE-YOLOv8 Training on RSNA 2024 Axial T2 Dataset

## Overview
This notebook trains the **Custom GE-YOLOv8** model on the **RSNA 2024 Lumbar Spine Degenerative Classification** dataset.

The model architecture is based on the paper *"Deep learning-based automatic detection and grading of disk herniation"* and includes:

**Key Features:**
- Custom CSP (Gradient Search) module for improved feature extraction
- ECAAttention mechanism for channel attention
- Identical data pipeline to YOLOv11 benchmark for fair comparison
- Training on Axial T2 MRI slices

**Repository**: [Yolov8-GS-ECA](https://github.com/hxxbb/Yolov8-GS-ECA)

## 1. Environment Setup

Install the custom GE-YOLOv8 model with CSP and ECAAttention modules.

In [ ]:
# Install required packages
!pip install -q pydicom opencv-python-headless pandas numpy pillow tqdm pyyaml

In [ ]:
# Clone the custom GE-YOLOv8 repository
# This repository contains custom modules: CSP (Gradient Search) and ECAAttention
# Note: This repo doesn't have setup.py, so we'll use standard ultralytics and add to path
!git clone https://github.com/hxxbb/Yolov8-GS-ECA.git

# Install standard ultralytics package
!pip install -q ultralytics

# Add the cloned repo to Python path so custom modules can be imported
import sys
sys.path.insert(0, '/kaggle/working/Yolov8-GS-ECA')

In [ ]:
# Verify installation and custom modules
import sys
import os

# Check that our custom repo is in the path
custom_repo_path = '/kaggle/working/Yolov8-GS-ECA'
if custom_repo_path in sys.path:
    print(f"✓ Custom repo added to Python path: {custom_repo_path}")
else:
    print(f"✗ Warning: Custom repo not in path")

# Verify the custom modules exist in the cloned repo
if os.path.exists(f'{custom_repo_path}/nn/modules/block.py'):
    print("✓ CSP module file found in custom repo")
else:
    print("✗ CSP module file not found")

if os.path.exists(f'{custom_repo_path}/nn/modules/Attention.py'):
    print("✓ ECAAttention module file found in custom repo")
else:
    print("✗ ECAAttention module file not found")

# Import ultralytics to verify it's installed
import ultralytics
print(f"\n✓ Ultralytics version: {ultralytics.__version__}")
print(f"✓ Ultralytics path: {ultralytics.__file__}")

print("\n⚠️  Important: The model will load custom modules from the cloned repo.")
print("   Make sure the YAML config path points to the cloned repo.")

## 2. Data Pipeline - RSNA 2024 Axial T2 Processing

**This pipeline must be IDENTICAL to the YOLOv11 experiment for fair comparison.**

### Data Pipeline Steps:
1. Load RSNA 2024 dataset (Axial T2 series only)
2. Process DICOM images to 8-bit
3. Resize to 384x384 (same as YOLOv11 benchmark)
4. Convert CSV annotations to bounding boxes
5. Create 3-class labels:
   - Class 0: Spinal Canal Stenosis (Center)
   - Class 1: Left Neural Foraminal & Left Subarticular (Left)
   - Class 2: Right Neural Foraminal & Right Subarticular (Right)
6. Propagate labels to adjacent slices (n-1, n+1)

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import pydicom
from pathlib import Path
from tqdm import tqdm
import yaml
from PIL import Image

# Configuration
DATASET_PATH = '/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification'
OUTPUT_PATH = '/kaggle/working/datasets/axial_t2'
IMAGE_SIZE = 384  # Same as YOLOv11 benchmark
BBOX_SIZE = 32    # Fixed bounding box size in pixels

# Create output directories
for split in ['train', 'val']:
    os.makedirs(f'{OUTPUT_PATH}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUTPUT_PATH}/labels/{split}', exist_ok=True)

print(f"Output directory created: {OUTPUT_PATH}")

In [ ]:
# Load annotations
train_df = pd.read_csv(f'{DATASET_PATH}/train.csv')
coords_df = pd.read_csv(f'{DATASET_PATH}/train_label_coordinates.csv')
series_desc_df = pd.read_csv(f'{DATASET_PATH}/train_series_descriptions.csv')

print(f"Total training samples: {len(train_df)}")
print(f"Total coordinate labels: {len(coords_df)}")
print(f"\nSeries descriptions:")
print(series_desc_df['series_description'].value_counts())

In [ ]:
# Filter for Axial T2 series only
axial_t2_series = series_desc_df[series_desc_df['series_description'] == 'Axial T2']
axial_t2_coords = coords_df[coords_df['series_id'].isin(axial_t2_series['series_id'])]

print(f"Axial T2 series count: {len(axial_t2_series)}")
print(f"Axial T2 coordinate labels: {len(axial_t2_coords)}")
print(f"\nConditions in Axial T2 data:")
print(axial_t2_coords['condition'].value_counts())

In [ ]:
def normalize_dicom_to_8bit(dicom_array):
    """Normalize DICOM pixel data to 8-bit grayscale."""
    # Handle different data types
    if dicom_array.dtype != np.uint8:
        # Normalize to 0-255 range
        min_val = dicom_array.min()
        max_val = dicom_array.max()
        if max_val > min_val:
            normalized = ((dicom_array - min_val) / (max_val - min_val) * 255.0).astype(np.uint8)
        else:
            normalized = np.zeros_like(dicom_array, dtype=np.uint8)
    else:
        normalized = dicom_array
    return normalized

def create_bbox_from_point(x, y, img_width, img_height, bbox_size=BBOX_SIZE):
    """Create a fixed-size bounding box centered at (x, y)."""
    half_size = bbox_size / 2
    
    # Calculate bbox in pixel coordinates
    x_min = max(0, x - half_size)
    y_min = max(0, y - half_size)
    x_max = min(img_width, x + half_size)
    y_max = min(img_height, y + half_size)
    
    # Convert to YOLO format (normalized: center_x, center_y, width, height)
    center_x = (x_min + x_max) / 2 / img_width
    center_y = (y_min + y_max) / 2 / img_height
    width = (x_max - x_min) / img_width
    height = (y_max - y_min) / img_height
    
    return center_x, center_y, width, height

def map_condition_to_class(condition):
    """Map RSNA conditions to 3 classes."""
    # Class 0: Spinal Canal Stenosis (Center)
    if 'Spinal Canal Stenosis' in condition:
        return 0
    # Class 1: Left side conditions
    elif 'Left Neural Foraminal' in condition or 'Left Subarticular' in condition:
        return 1
    # Class 2: Right side conditions
    elif 'Right Neural Foraminal' in condition or 'Right Subarticular' in condition:
        return 2
    else:
        return None  # Unknown condition

print("Helper functions defined successfully.")

In [ ]:
def process_series(series_id, study_id, instance_number, coords_for_series, output_split='train'):
    """Process a single DICOM series and create YOLO format labels."""
    # Load DICOM file
    dicom_path = f'{DATASET_PATH}/train_images/{study_id}/{series_id}/{instance_number}.dcm'
    
    if not os.path.exists(dicom_path):
        return False
    
    try:
        # Read DICOM
        dcm = pydicom.dcmread(dicom_path)
        img_array = dcm.pixel_array
        
        # Normalize to 8-bit
        img_8bit = normalize_dicom_to_8bit(img_array)
        
        # Convert to 3-channel for consistency (RGB)
        img_rgb = cv2.cvtColor(img_8bit, cv2.COLOR_GRAY2RGB)
        
        # Resize to target size
        img_resized = cv2.resize(img_rgb, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
        
        # Calculate resize factors for coordinate adjustment
        height_factor = IMAGE_SIZE / img_array.shape[0]
        width_factor = IMAGE_SIZE / img_array.shape[1]
        
        # Create labels
        labels = []
        for _, row in coords_for_series.iterrows():
            class_id = map_condition_to_class(row['condition'])
            if class_id is None:
                continue
            
            # Adjust coordinates for resizing
            x = row['x'] * width_factor
            y = row['y'] * height_factor
            
            # Create bounding box
            center_x, center_y, width, height = create_bbox_from_point(x, y, IMAGE_SIZE, IMAGE_SIZE)
            
            # YOLO format: class_id center_x center_y width height (all normalized)
            labels.append(f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}")
        
        # Save image and labels
        img_filename = f"{study_id}_{series_id}_{instance_number}.jpg"
        img_path = f'{OUTPUT_PATH}/images/{output_split}/{img_filename}'
        label_path = f'{OUTPUT_PATH}/labels/{output_split}/{study_id}_{series_id}_{instance_number}.txt'
        
        # Save image
        cv2.imwrite(img_path, cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR))
        
        # Save labels
        with open(label_path, 'w') as f:
            f.write('\n'.join(labels))
        
        return True
    
    except Exception as e:
        print(f"Error processing {dicom_path}: {e}")
        return False

print("Series processing function defined.")

In [ ]:
# Process all Axial T2 images
print("Processing Axial T2 images...\n")

processed_count = 0
train_count = 0
val_count = 0

# Group coordinates by series_id and instance_number
grouped = axial_t2_coords.groupby(['study_id', 'series_id', 'instance_number'])

# Split into train/val (80/20)
unique_studies = axial_t2_coords['study_id'].unique()
np.random.seed(42)
np.random.shuffle(unique_studies)
split_idx = int(len(unique_studies) * 0.8)
train_studies = set(unique_studies[:split_idx])
val_studies = set(unique_studies[split_idx:])

for (study_id, series_id, instance_number), coords_group in tqdm(grouped, desc="Processing images"):
    # Determine split
    output_split = 'train' if study_id in train_studies else 'val'
    
    # Process current slice
    if process_series(series_id, study_id, instance_number, coords_group, output_split):
        processed_count += 1
        if output_split == 'train':
            train_count += 1
        else:
            val_count += 1
    
    # Process adjacent slices (n-1 and n+1) with same labels for shared coordinates
    for offset in [-1, 1]:
        adj_instance = instance_number + offset
        if adj_instance >= 0:  # Skip negative instance numbers
            if process_series(series_id, study_id, adj_instance, coords_group, output_split):
                processed_count += 1
                if output_split == 'train':
                    train_count += 1
                else:
                    val_count += 1

print(f"\nProcessing complete!")
print(f"Total images processed: {processed_count}")
print(f"Training images: {train_count}")
print(f"Validation images: {val_count}")

In [ ]:
# Create data.yaml for YOLO training
data_yaml = {
    'path': OUTPUT_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 3,  # Number of classes
    'names': [
        'Spinal_Canal_Stenosis',      # Class 0: Center
        'Left_Neural_Subarticular',   # Class 1: Left side
        'Right_Neural_Subarticular'   # Class 2: Right side
    ]
}

yaml_path = f'{OUTPUT_PATH}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"Data configuration saved to: {yaml_path}")
print("\nDataset structure:")
print(yaml.dump(data_yaml, sort_keys=False))

## 3. Model Configuration - Custom GE-YOLOv8

Load the custom YOLOv8-ECA-CSP architecture with:
- **CSP modules** (Gradient Search) for improved feature extraction
- **ECAAttention** for channel attention mechanism

In [ ]:
import os

# Path to custom model configuration
model_config_path = '/kaggle/working/Yolov8-GS-ECA/models/v8/yolov8-ECA-CSP.yaml'

# Verify the config file exists
if os.path.exists(model_config_path):
    print(f"✓ Model config found: {model_config_path}")
    
    # Read and display config
    with open(model_config_path, 'r') as f:
        config_content = f.read()
    print("\nModel Configuration:")
    print("=" * 50)
    print(config_content[:500])  # Display first 500 chars
    print("...")
else:
    print(f"✗ Model config NOT found at: {model_config_path}")
    print("Please verify the repository was cloned correctly.")


In [ ]:
# Initialize the custom GE-YOLOv8 model from YAML
# The key insight: ultralytics uses globals() to look up modules when parsing YAML
# Additional: The custom repo uses old ultralytics.yolo imports that need compatibility patches

import sys
import os

# Add custom repo to path FIRST
custom_repo = '/kaggle/working/Yolov8-GS-ECA'
if custom_repo not in sys.path:
    sys.path.insert(0, custom_repo)

# CRITICAL: Create compatibility layer for old ultralytics.yolo imports
# The custom GE-YOLOv8 repo was built for an older ultralytics version that used
# 'ultralytics.yolo.utils' which is now 'ultralytics.utils'
print("Setting up compatibility layer for ultralytics imports...")
import ultralytics

# Create the old module structure as an alias to the new one
if not hasattr(ultralytics, 'yolo'):
    # Create a mock yolo module that redirects to the correct locations
    class YoloCompatModule:
        pass
    
    ultralytics.yolo = YoloCompatModule()
    
    # Map old paths to new paths
    import ultralytics.utils
    ultralytics.yolo.utils = ultralytics.utils
    
    print("✓ Created ultralytics.yolo compatibility layer")

# Now we can safely import custom modules
try:
    from nn.modules.block import CSP, Bottleneck, C2f
    from nn.modules.Attention import ECAAttention
    print("✓ Custom modules imported successfully")
    print(f"  - CSP: {CSP}")
    print(f"  - ECAAttention: {ECAAttention}")
except ImportError as e:
    print(f"⚠️  Error importing custom modules: {e}")
    import traceback
    traceback.print_exc()
    raise

# Now import YOLO and inject modules into the tasks module's globals
# This is where parse_model() looks up module names
from ultralytics import YOLO
import ultralytics.nn.tasks as tasks_module

# Inject custom modules into the tasks module's global namespace
# ultralytics.nn.tasks.parse_model uses globals()[module_name] to resolve modules
tasks_module.CSP = CSP
tasks_module.ECAAttention = ECAAttention

print("✓ Custom modules injected into ultralytics.nn.tasks")
print(f"  - tasks_module.CSP: {hasattr(tasks_module, 'CSP')}")
print(f"  - tasks_module.ECAAttention: {hasattr(tasks_module, 'ECAAttention')}")

print("\nInitializing GE-YOLOv8 model with custom architecture...")
print(f"Using config from: {model_config_path}")

# Build model from custom YAML
try:
    model = YOLO(model_config_path)
    print("\n" + "=" * 50)
    print("Model initialized successfully!")
    print("=" * 50)
    
    # Display model information
    model.info(verbose=True)
    
except KeyError as e:
    print(f"\n❌ KeyError: {e}")
    print("\nModule not found in ultralytics.nn.tasks globals.")
    print("Available custom modules:")
    print(f"  - CSP in tasks: {hasattr(tasks_module, 'CSP')}")
    print(f"  - ECAAttention in tasks: {hasattr(tasks_module, 'ECAAttention')}")
    raise
    
except Exception as e:
    print(f"\n❌ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    raise


## 4. Training - Benchmark Settings

Train with identical settings to YOLOv11 experiment for fair comparison:
- Image size: 384x384
- Epochs: 50
- Batch size: 16
- Optimizer: AdamW
- Learning rate: 0.001

In [ ]:
# Training configuration (matching YOLOv11 benchmark)
training_config = {
    'data': yaml_path,
    'epochs': 50,
    'imgsz': 384,
    'batch': 16,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'device': 0,  # Use GPU 0
    'workers': 8,
    'project': '/kaggle/working/ge_yolov8_benchmark',
    'name': 'axial_t2_run',
    'exist_ok': True,
    'pretrained': False,  # Train from scratch with custom architecture
    'patience': 10,  # Early stopping patience
    'save': True,
    'save_period': 5,  # Save checkpoint every 5 epochs
    'cache': False,  # Don't cache images (save memory)
    'verbose': True
}

print("Training Configuration:")
print("=" * 50)
for key, value in training_config.items():
    print(f"{key:15} : {value}")
print("=" * 50)

In [ ]:
# Start training
print("\n" + "=" * 70)
print("STARTING GE-YOLOV8 TRAINING")
print("=" * 70 + "\n")

results = model.train(**training_config)

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print("=" * 70)

## 5. Results and Comparison

Extract and display key metrics for comparison with YOLOv11 benchmark.

In [ ]:
# Load the best model
best_model_path = '/kaggle/working/ge_yolov8_benchmark/axial_t2_run/weights/best.pt'
best_model = YOLO(best_model_path)

print("Loaded best model for evaluation.")

In [ ]:
# Validate on validation set
print("\n" + "=" * 70)
print("VALIDATION RESULTS")
print("=" * 70 + "\n")

val_results = best_model.val(data=yaml_path, imgsz=384, batch=16)

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

In [ ]:
# Extract and display key metrics
print("\n" + "=" * 70)
print("GE-YOLOV8 BENCHMARK METRICS - RSNA 2024 AXIAL T2")
print("=" * 70)
print()

# Check if validation results exist
if val_results is None:
    print("⚠️  Warning: Validation results not available")
else:
    # Access box metrics - handle different ultralytics versions
    try:
        # Try to access box metrics
        if hasattr(val_results, 'box'):
            box_metrics = val_results.box
        elif hasattr(val_results, 'results_dict'):
            box_metrics = val_results.results_dict
        else:
            box_metrics = val_results
        
        print("📊 DETECTION METRICS (Box):")
        print("-" * 70)
        
        # Safely access metrics with fallbacks
        def safe_get_metric(obj, attr_names, default=0.0):
            """Try multiple attribute names and return default if none exist."""
            if not isinstance(attr_names, list):
                attr_names = [attr_names]
            for attr in attr_names:
                if hasattr(obj, attr):
                    return getattr(obj, attr)
                elif isinstance(obj, dict) and attr in obj:
                    return obj[attr]
            return default
        
        map50 = safe_get_metric(box_metrics, ['map50', 'metrics/mAP50(B)'])
        map50_95 = safe_get_metric(box_metrics, ['map', 'map50_95', 'metrics/mAP50-95(B)'])
        map75 = safe_get_metric(box_metrics, ['map75', 'metrics/mAP75(B)'])
        precision = safe_get_metric(box_metrics, ['mp', 'metrics/precision(B)'])
        recall = safe_get_metric(box_metrics, ['mr', 'metrics/recall(B)'])
        
        print(f"  mAP50        : {map50:.4f}")
        print(f"  mAP50-95     : {map50_95:.4f}")
        print(f"  mAP75        : {map75:.4f}")
        print(f"  Precision    : {precision:.4f}")
        print(f"  Recall       : {recall:.4f}")
        print()
        
        # Per-class metrics
        print("📋 PER-CLASS PERFORMANCE:")
        print("-" * 70)
        class_names = ['Spinal_Canal', 'Left_Neural', 'Right_Neural']
        
        # Try to get per-class AP50
        ap50_per_class = safe_get_metric(box_metrics, ['ap50', 'ap_class_index'], None)
        if ap50_per_class is not None and hasattr(ap50_per_class, '__iter__'):
            for i, class_name in enumerate(class_names):
                if i < len(ap50_per_class):
                    print(f"  Class {i} ({class_name:20}): mAP50 = {ap50_per_class[i]:.4f}")
        else:
            print("  Per-class metrics not available in this format")
        print()
        
        print("=" * 70)
        print()
        
        # Summary for comparison
        print("\n📈 SUMMARY FOR YOLOV11 COMPARISON:")
        print("=" * 70)
        print(f"Model           : GE-YOLOv8 (CSP + ECAAttention)")
        print(f"Dataset         : RSNA 2024 - Axial T2 (384x384)")
        print(f"Classes         : 3 (Spinal Canal, Left Neural, Right Neural)")
        print(f"Training Images : {train_count}")
        print(f"Val Images      : {val_count}")
        print(f"Epochs          : 50")
        print(f"Batch Size      : 16")
        print(f"Optimizer       : AdamW (lr=0.001)")
        print()
        print(f"mAP50           : {map50:.4f}  ← Compare with YOLOv11")
        print(f"mAP50-95        : {map50_95:.4f}  ← Compare with YOLOv11")
        print("=" * 70)
        
    except Exception as e:
        print(f"⚠️  Error accessing validation metrics: {e}")
        print(f"\nAvailable attributes: {dir(val_results)}")
        print("\nTip: Check ultralytics version and results structure")


In [ ]:
# Display training curves (if available)
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path('/kaggle/working/ge_yolov8_benchmark/axial_t2_run')

# Check for results.csv or results.png
results_csv = results_dir / 'results.csv'
results_png = results_dir / 'results.png'

if results_csv.exists():
    print("\n📊 Training curves available at:")
    print(f"   CSV: {results_csv}")
    
    # Load and display results
    results_df = pd.read_csv(results_csv)
    print(f"\nTraining history (last 5 epochs):")
    print(results_df.tail())

if results_png.exists():
    print(f"\n   Plot: {results_png}")
    # Display the plot
    img = Image.open(results_png)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('GE-YOLOv8 Training Results')
    plt.show()

In [ ]:
# Save final summary
summary_path = '/kaggle/working/ge_yolov8_benchmark_summary.txt'

# Safely get metrics
def get_metric_safe(results, attr_names, default='N/A'):
    """Safely extract metric from results."""
    if results is None:
        return default
    
    try:
        if hasattr(results, 'box'):
            obj = results.box
        elif hasattr(results, 'results_dict'):
            obj = results.results_dict
        else:
            obj = results
        
        if not isinstance(attr_names, list):
            attr_names = [attr_names]
        
        for attr in attr_names:
            if hasattr(obj, attr):
                val = getattr(obj, attr)
                return f"{val:.4f}" if isinstance(val, (int, float)) else str(val)
            elif isinstance(obj, dict) and attr in obj:
                val = obj[attr]
                return f"{val:.4f}" if isinstance(val, (int, float)) else str(val)
    except Exception as e:
        print(f"Warning: Could not extract metric {attr_names}: {e}")
    
    return default

map50_val = get_metric_safe(val_results, ['map50', 'metrics/mAP50(B)'])
map50_95_val = get_metric_safe(val_results, ['map', 'map50_95', 'metrics/mAP50-95(B)'])
map75_val = get_metric_safe(val_results, ['map75', 'metrics/mAP75(B)'])
precision_val = get_metric_safe(val_results, ['mp', 'metrics/precision(B)'])
recall_val = get_metric_safe(val_results, ['mr', 'metrics/recall(B)'])

with open(summary_path, 'w') as f:
    f.write("GE-YOLOV8 BENCHMARK RESULTS - RSNA 2024 AXIAL T2\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Model Architecture: GE-YOLOv8 (CSP + ECAAttention)\n")
    f.write(f"Dataset: RSNA 2024 Lumbar Spine - Axial T2 MRI\n")
    f.write(f"Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}\n")
    f.write(f"Number of Classes: 3\n")
    f.write(f"Training Samples: {train_count}\n")
    f.write(f"Validation Samples: {val_count}\n")
    f.write(f"\nTraining Configuration:\n")
    f.write(f"  Epochs: 50\n")
    f.write(f"  Batch Size: 16\n")
    f.write(f"  Optimizer: AdamW\n")
    f.write(f"  Learning Rate: 0.001\n")
    f.write(f"\nValidation Metrics:\n")
    f.write(f"  mAP50: {map50_val}\n")
    f.write(f"  mAP50-95: {map50_95_val}\n")
    f.write(f"  mAP75: {map75_val}\n")
    f.write(f"  Precision: {precision_val}\n")
    f.write(f"  Recall: {recall_val}\n")
    f.write(f"\nModel Weights: {best_model_path}\n")

print(f"\n✓ Summary saved to: {summary_path}")
print("\nBenchmarking complete! Compare these metrics with your YOLOv11 results.")


## Conclusion

This notebook trained the custom **GE-YOLOv8** model with:
- ✅ Custom CSP (Gradient Search) modules
- ✅ ECAAttention mechanism
- ✅ Identical data pipeline to YOLOv11 (384x384, 3 classes)
- ✅ Same training configuration (50 epochs, batch=16, AdamW)

**Next Steps:**
1. Compare mAP50 and mAP50-95 with YOLOv11 baseline
2. Analyze per-class performance differences
3. Consider ensemble methods if appropriate

**Files Generated:**
- Model weights: `/kaggle/working/ge_yolov8_benchmark/axial_t2_run/weights/best.pt`
- Summary: `/kaggle/working/ge_yolov8_benchmark_summary.txt`
- Dataset: `/kaggle/working/datasets/axial_t2/`